# 統一Factory実践：モデルから候補点提案まで

このNotebookで確認すること：公開名の探索、既存クラスとの同一性、明示的な学習、posterior、獲得関数、OptimizerとSearch Strategyの違い、Mixed・制約・多目的・分類ALへの応用。

[Theory](../../docs/theory/13_model_selection.md) / [Guide](../../docs/guide/13_model_selection.md) / [利用方法](../../docs/capability_workflow.md)

Data → Model → Fit → Posterior → Acquisition → Optimization → Candidate → Evaluation

合成データを使用し、回帰BOでは最大化を行います。分類ALでは目的値ではなくラベルの不確実性を評価します。

In [ ]:
import torch
import matplotlib.pyplot as plt
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.sampling.normal import SobolQMCNormalSampler
from botorch.utils.multi_objective.box_decompositions.non_dominated import (
    FastNondominatedPartitioning,
)
from robotorchan.models import SingleTaskGP, create_model, get_model_class, list_models
from robotorchan.acquisition import (
    create_acquisition, get_acquisition_class, list_acquisitions,
)
from robotorchan.optim import (
    create_optimizer, create_search_strategy, list_optimizers, list_search_strategies,
)
from robotorchan.optim.constraints.contracts import CandidateConstraints
from robotorchan.optim.domains.variable_space import MixedVariableSpace

torch.manual_seed(17)
dtype = torch.double

## 1. 公開名とコンストラクタ

一覧は実装の発見に使います。名前の存在だけでは組合せの実行可能性を保証しません。Factoryは既存のクラスを返し、独自の代理モデルを作りません。

In [ ]:
for names in (list_models(), list_acquisitions(), list_optimizers(), list_search_strategies()):
    print(len(names), list(names)[:8])
assert get_model_class("SingleTaskGP") is SingleTaskGP
assert get_acquisition_class("qLogExpectedImprovement") is qLogExpectedImprovement
print(list_optimizers())
print(list_search_strategies())

## 2. 単目的回帰：データ・学習・予測

入力は n × d、観測は n × 1。真の目的関数は $f(x)=-(x-0.65)^2$ です。Factoryはfitを呼びません。直接生成との型の一致を確認してから学習します。

In [ ]:
X = torch.linspace(0.1, 0.9, 9, dtype=dtype).unsqueeze(-1)
Y = -((X - 0.65) ** 2)
bounds = torch.tensor([[0.0], [1.0]], dtype=dtype)
model = create_model("SingleTaskGP", X, Y)
direct = SingleTaskGP(X, Y)
assert type(model) is type(direct)
model.fit(optimizer_kwargs={"options": {"maxiter": 30}})
grid = torch.linspace(0, 1, 101, dtype=dtype).unsqueeze(-1)
with torch.no_grad():
    posterior = model.posterior(grid)
    mean = posterior.mean.squeeze(-1)
    std = posterior.variance.clamp_min(0).sqrt().squeeze(-1)
print(X.shape, Y.shape, posterior.mean.shape, posterior.variance.shape)

## 3. 獲得関数とq-batch

best_fは既観測の最大値。qLogExpectedImprovementは改善量の期待値を安定な対数表現で扱うMC獲得関数です。評価入力は batch × q × d。q=1の曲線とq=2の同時提案は別の計算です。

In [ ]:
acq = create_acquisition(
    "qLogExpectedImprovement", model=model, best_f=Y.max(),
    sampler=SobolQMCNormalSampler(torch.Size([32]), seed=17),
)
with torch.no_grad():
    scores = acq(grid.unsqueeze(-2))
print(scores.shape)
optimize = create_optimizer("sobol", raw_samples=64, seed=17)
candidate, value = optimize(acq, bounds, q=2)
assert candidate.shape == (2, 1)
assert torch.isfinite(candidate).all()
print(candidate, value, -((candidate - 0.65) ** 2))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
axes[0].plot(grid.squeeze(), mean, label="Posterior mean")
axes[0].fill_between(grid.squeeze(), mean - 1.96 * std, mean + 1.96 * std, alpha=0.2)
axes[0].scatter(X.squeeze(), Y.squeeze(), label="Observations")
axes[0].scatter(candidate.squeeze(), -((candidate.squeeze() - 0.65) ** 2), label="Candidates")
axes[0].legend()
axes[1].plot(grid.squeeze(), scores, label="q=1 log EI")
axes[1].legend()
axes[1].set_xlabel("x")
plt.tight_layout()
plt.show()

## 4. Search Strategy Factory

Optimizer Factoryは関数を返すのに対し、Search Strategy Factoryは探索空間を持つインスタンスを返します。ここではSobolを使い、同じ獲得関数から候補を生成します。

In [ ]:
strategy = create_search_strategy("SobolSearchStrategy", bounds, num_samples=64, seed=17)
result = strategy.optimize(acq, q=2)
assert result.candidates.shape == (2, 1)
print(result.candidates, result.acquisition_value)

## 5. Mixed入力と候補制約

第2変数は整数0/1。モデルのcat_dimsはカテゴリカーネルの指定であり、Optimizer側のinteger_dimsは候補の許容値を指定します。線形不等式は x[0] >= 0.2 です。

In [ ]:
Xm = torch.cat([X, (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1)], dim=-1)
Ym = Y + 0.1 * Xm[:, 1:]
bm = torch.tensor([[0., 0.], [1., 1.]], dtype=dtype)
mixed = create_model("MixedSingleTaskGP", Xm, Ym, cat_dims=[1])
mixed.fit(optimizer_kwargs={"options": {"maxiter": 10}})
am = create_acquisition("qLogExpectedImprovement", model=mixed, best_f=Ym.max())
space = MixedVariableSpace(bounds=bm, integer_dims=(1,))
constraints = CandidateConstraints(
    inequality_constraints=((torch.tensor([0]), torch.tensor([1.], dtype=dtype), 0.2),),
)
cm, vm = optimize(am, bm, q=2, variable_space=space, constraints=constraints)
assert cm.shape == (2, 2)
assert torch.all(cm[:, 0] >= 0.2 - 1e-8)
assert torch.equal(cm[:, 1], cm[:, 1].round())
print(cm, vm)

## 6. 多目的回帰

2目的を同時に最大化します。参照点は観測の各目的の最小値より低く設定します。qLogEHVIはPareto集合のhypervolume改善を評価します。

In [ ]:
Y2 = torch.cat([Y, -((X - 0.25) ** 2)], dim=-1)
m2 = create_model("SingleTaskGP", X, Y2)
m2.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ref = Y2.min(dim=0).values - 0.1
partition = FastNondominatedPartitioning(ref_point=ref, Y=Y2)
a2 = create_acquisition(
    "qLogExpectedHypervolumeImprovement", model=m2, ref_point=ref.tolist(),
    partitioning=partition, sampler=SobolQMCNormalSampler(torch.Size([16]), seed=17),
)
c2, v2 = optimize(a2, bounds, q=2)
y2_new = torch.cat([-((c2 - 0.65) ** 2), -((c2 - 0.25) ** 2)], dim=-1)
assert c2.shape == (2, 1)
print(c2, v2, y2_new)
plt.scatter(Y2[:, 0], Y2[:, 1], label="Observations")
plt.scatter(y2_new[:, 0], y2_new[:, 1], label="Candidates")
plt.xlabel("Objective 1")
plt.ylabel("Objective 2")
plt.legend()
plt.show()

## 7. 二値・多クラス・順序分類のAL

クラスラベルを数値回帰へ変換しません。各分類モデルをfitし、predict_probaで予測クラス確率を確認します。qPredictiveEntropyはラベルの不確実性に基づく獲得関数で、目的値の最大化とは異なります。実データでは検証データを用いて学習回数や予測確率の較正を評価してください。

In [ ]:
for name, k in [
    ("BinarySingleTaskGPClassifier", 2),
    ("MulticlassSingleTaskGPClassifier", 3),
    ("OrdinalSingleTaskGPClassifier", 3),
]:
    labels = (X.squeeze(-1) * k).long()
    kwargs = {"num_classes": k} if k > 2 else {}
    classifier = create_model(name, X, labels, **kwargs)
    classifier.fit(epochs=2)
    with torch.no_grad():
        probabilities = classifier.predict_proba(grid)
    entropy = create_acquisition("qPredictiveEntropy", model=classifier)
    proposed = strategy.optimize(entropy, q=2).candidates
    assert proposed.shape == (2, 1)
    print(name, probabilities.shape, proposed.flatten().tolist())
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), probabilities)
    for x in proposed.flatten():
        plt.axvline(x.item(), linestyle="--")
    plt.title(name)
    plt.xlabel("x")
    plt.ylabel("Class probability")
    plt.show()

## 8. 観測制約付きBO

出力を目的値と制約値の2列に分けます。GenericMCObjectiveで第1出力を目的とし、第2出力が0以下となる確率を獲得関数に反映します。候補制約とは異なり、観測制約の成立は実験まで確定しません。

In [ ]:
from botorch.acquisition.objective import GenericMCObjective

Yc = torch.cat([Y, X - 0.8], dim=-1)
mc = create_model("SingleTaskGP", X, Yc)
mc.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ac = create_acquisition(
    "qLogExpectedImprovement", model=mc, best_f=Y[X[:, 0] <= 0.8].max(),
    objective=GenericMCObjective(lambda samples, X=None: samples[..., 0]),
    constraints=[lambda samples: samples[..., 1]],
)
cc, vc = optimize(ac, bounds, q=2)
assert cc.shape == (2, 1)
print(cc, vc, "observed constraint:", cc - 0.8)
with torch.no_grad():
    pc = mc.posterior(grid)
plt.plot(grid.squeeze(), pc.mean[:, 1], label="Constraint mean")
plt.axhline(0, linestyle="--")
plt.scatter(cc.squeeze(), cc.squeeze() - 0.8, label="Evaluated candidates")
plt.legend()
plt.show()

## 9. Variational GP

Exact GPと異なり、誘導点を用いた変分推論を行います。fitの引数も異なります。Factoryはこれらを共通化せず、元のコンストラクタと学習契約を維持します。

In [ ]:
vgp = create_model("SingleTaskVariationalGP", X, Y, inducing_points=5)
vgp.fit(epochs=10, lr=0.03, batch_size=9)
vgp.eval()
vgp.likelihood.eval()
with torch.no_grad():
    vp = vgp.posterior(grid)
va = create_acquisition("qLogExpectedImprovement", model=vgp, best_f=Y.max())
cv, vv = optimize(va, bounds, q=2)
assert cv.shape == (2, 1)
print(cv, vv, -((cv - 0.65) ** 2))
plt.plot(grid.squeeze(), vp.mean.squeeze(), label="Variational GP")
plt.scatter(X.squeeze(), Y.squeeze())
plt.scatter(cv.squeeze(), -((cv.squeeze() - 0.65) ** 2))
plt.legend()
plt.show()

## 10. Multi-Fidelity GP：高忠実度での候補提案

第2列は忠実度です。低忠実度には系統的な偏りを加えます。ここでは最終忠実度1に固定した候補提案を示します。評価コストを考慮して忠実度自体を選ぶMFKGとは区別します。

In [ ]:
fidelity = (torch.arange(len(X)) % 2).to(dtype).unsqueeze(-1) * 0.5 + 0.5
Xf = torch.cat([X, fidelity], dim=-1)
Yf = Y + 0.2 * (1 - fidelity)
mf = create_model("SingleTaskMultiFidelityGP", Xf, Yf, data_fidelities=[1])
mf.fit(optimizer_kwargs={"options": {"maxiter": 10}})
af = create_acquisition("PosteriorMean", model=mf)
bf = torch.tensor([[0., 0.5], [1., 1.]], dtype=dtype)
native_opt = create_optimizer("botorch", num_restarts=2, raw_samples=16)
cf, vf = native_opt(af, bf, q=1, fixed_features={1: 1.0})
assert cf.shape == (1, 2)
assert cf[0, 1].item() == 1.0
print(cf, vf, -((cf[:, :1] - 0.65) ** 2))
with torch.no_grad():
    for f in (0.5, 1.0):
        pg = mf.posterior(torch.cat([grid, torch.full_like(grid, f)], dim=-1))
        plt.plot(grid.squeeze(), pg.mean.squeeze(), label=f"Fidelity {f}")
plt.legend()
plt.show()

## 11. Multi-Task GP：対象タスクを指定する

task_featureは学習入力のタスク列、output_tasksは予測対象です。対象タスクの候補はタスク列を含まない物理変数空間で生成します。

In [ ]:
Xt = torch.cat([
    torch.cat([X, torch.zeros_like(X)], dim=-1),
    torch.cat([X, torch.ones_like(X)], dim=-1),
])
Yt = torch.cat([Y + 0.1, -((X - 0.4) ** 2)])
mt = create_model("MultiTaskGP", Xt, Yt, task_feature=1, output_tasks=[1])
mt.fit(optimizer_kwargs={"options": {"maxiter": 10}})
at = create_acquisition("qLogExpectedImprovement", model=mt, best_f=Yt[len(X):].max())
ct, vt = native_opt(at, bounds, q=1)
assert ct.shape == (1, 1)
print(ct, vt, -((ct - 0.4) ** 2))
with torch.no_grad():
    pt = mt.posterior(grid)
plt.plot(grid.squeeze(), pt.mean.squeeze(), label="Target task")
plt.scatter(X.squeeze(), Yt[len(X):].squeeze())
plt.scatter(ct.squeeze(), -((ct.squeeze() - 0.4) ** 2))
plt.legend()
plt.show()

## 12. Preference BO：比較データから候補ペアを提案

観測値は実数ではなく勝者・敗者のインデックスです。AnalyticExpectedUtilityOfBestOptionは2候補のうち好まれる方の潜在効用を評価します。

In [ ]:
datapoints = X.clone()
pairs = [(i, i + 1) if Y[i] >= Y[i + 1] else (i + 1, i) for i in range(len(X) - 1)]
comparisons = torch.tensor(pairs, dtype=torch.long)
pref = create_model("PairwiseGP", datapoints=datapoints, comparisons=comparisons)
pref.fit(optimizer_kwargs={"options": {"maxiter": 10}})
ap = create_acquisition("AnalyticExpectedUtilityOfBestOption", pref_model=pref)
cp, vp = native_opt(ap, bounds, q=2)
assert cp.shape == (2, 1)
print(cp, vp, "preferred index:", (-((cp - 0.65) ** 2)).argmax().item())
with torch.no_grad():
    pp = pref.posterior(grid)
plt.plot(grid.squeeze(), pp.mean.squeeze(), label="Latent utility")
for x in cp.flatten():
    plt.axvline(x.item(), linestyle="--")
plt.legend()
plt.show()

## 13. 獲得関数とOptimizerの変更

同じ学習済み回帰GPに対し、LogEI、UCB、qLogNEIを生成して候補を比較します。qLogNEIには観測入力X_baselineを渡します。獲得値の尺度は異なるため、異なる獲得関数間で数値を直接比較しません。

In [ ]:
for name, kwargs in [
    ("LogExpectedImprovement", {"best_f": Y.max()}),
    ("UpperConfidenceBound", {"beta": 2.0}),
    ("qLogNoisyExpectedImprovement", {"X_baseline": X}),
]:
    acquisition = create_acquisition(name, model=model, **kwargs)
    point, score = native_opt(acquisition, bounds, q=1)
    assert point.shape == (1, 1)
    print(name, point, score, -((point - 0.65) ** 2))

## 14. Factoryによる反復BO

各反復で観測データを追加し、モデルと獲得関数を再構築します。Studyを使わない明示的なループです。

In [ ]:
loop_X, loop_Y = X.clone(), Y.clone()
history = [loop_Y.max().item()]
for _ in range(2):
    loop_model = create_model("SingleTaskGP", loop_X, loop_Y)
    loop_model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
    loop_acq = create_acquisition("qLogExpectedImprovement", model=loop_model, best_f=loop_Y.max())
    point, _ = optimize(loop_acq, bounds, q=2)
    loop_X = torch.cat([loop_X, point.detach()])
    loop_Y = torch.cat([loop_Y, -((point.detach() - 0.65) ** 2)])
    history.append(loop_Y.max().item())
assert len(loop_X) == len(X) + 4
plt.plot(range(len(history)), history, marker="o")
plt.xlabel("BO iteration")
plt.ylabel("Best observed objective")
plt.show()

## 15. 次元削減・非定常・非GPモデル

PCAとPLSは低次元表現を利用しますが、候補は元の入力空間で生成します。非定常GPは入力位置に依存する変動を表現します。Random Forestは木の集合に基づく予測を利用するため、勾配を必要としない探索を選びます。

In [ ]:
Xh = torch.rand(20, 6, dtype=dtype)
Yh = -((Xh[:, :2] - 0.65) ** 2).sum(dim=-1, keepdim=True)
bh = torch.stack([torch.zeros(6, dtype=dtype), torch.ones(6, dtype=dtype)])
for name in ("PCAGP", "PLSGP", "RandomProjectionGP"):
    surrogate = create_model(name, Xh, Yh, n_components=2)
    acquisition_name = "qLogExpectedImprovement"
    optimizer_name = "sobol"
    surrogate.fit()
    with torch.no_grad():
        posterior = surrogate.posterior(Xh)
    acquisition = create_acquisition(acquisition_name, model=surrogate, best_f=Yh.max())
    solver = create_optimizer(optimizer_name, raw_samples=32, seed=17)
    point, value = solver(acquisition, bh, q=2)
    assert point.shape == (2, 6)
    print(name, posterior.mean.shape, point, value)
    print("Objective:", -((point[:, :2] - 0.65) ** 2).sum(dim=-1))

In [ ]:
Xn = torch.linspace(0, 1, 16, dtype=dtype).unsqueeze(-1)
Yn = torch.where(Xn < 0.5, torch.sin(2 * torch.pi * Xn), torch.sin(8 * torch.pi * Xn))
for name, kwargs, acquisition_name in [
    ("NonstationarySingleTaskGP", {}, "qLogExpectedImprovement"),
    ("RandomForestSurrogate", {"n_estimators": 32, "random_state": 17}, "qExpectedImprovement"),
]:
    surrogate = create_model(name, Xn, Yn, **kwargs)
    optimizer_name = "sobol"
    surrogate.fit()
    acquisition = create_acquisition(acquisition_name, model=surrogate, best_f=Yn.max())
    solver = create_optimizer(optimizer_name, raw_samples=32, seed=17)
    point, value = solver(acquisition, bounds, q=1)
    assert point.shape == (1, 1)
    with torch.no_grad():
        mean = surrogate.posterior(grid).mean.squeeze()
    plt.figure(figsize=(7, 3))
    plt.plot(grid.squeeze(), mean, label=name)
    plt.scatter(Xn.squeeze(), Yn.squeeze())
    plt.axvline(point.item(), linestyle="--")
    plt.legend()
    plt.show()
    print(point, value)

## 16. Optimizerと探索方式の比較

同じ学習済みGPと獲得関数を使用し、ランダム探索、Sobol探索、BoTorchの勾配ベース最適化を比較します。探索予算と初期化の違いがあるため、単一試行の獲得値だけで手法の優劣は判断できません。

In [ ]:
comparison_model = create_model("SingleTaskGP", X, Y)
comparison_model.fit(optimizer_kwargs={"options": {"maxiter": 10}})
comparison_acq = create_acquisition(
    "LogExpectedImprovement", model=comparison_model, best_f=Y.max()
)
for name in ("random", "sobol", "botorch"):
    
    solver = create_optimizer(name, raw_samples=32, num_restarts=2)
    point, value = solver(comparison_acq, bounds, q=1)
    assert point.shape == (1, 1)
    print(name, point.item(), value.item())
for name in ("RandomSearchStrategy", "SobolSearchStrategy"):
    
    search = create_search_strategy(name, bounds, num_samples=32, seed=17)
    result = search.optimize(comparison_acq, q=1)
    assert result.candidates.shape == (1, 1)
    print(name, result.candidates, result.acquisition_value)

## 17. 実務での構成選択と関連資料

Factoryは名前解決と引数の受け渡しを担当します。モデル固有の引数、posteriorの意味、制約、q対応は各APIの契約に従います。新しい組合せを利用する際は、モデルの学習、posteriorの形状、獲得関数の要件、候補の実行可能性を順に確認してください。

問題要件から構成を絞り込むには [Capability・Recommendation](63_capability_recommendation.ipynb)、反復実験の試行管理には [Study API](46_study_ask_tell_mixed_q_batch.ipynb) が利用できます。

参考：Rasmussen and Williams (2006), Gaussian Processes for Machine Learning; Frazier (2018), A Tutorial on Bayesian Optimization.